In [ ]:
# Cluster-side dependencies: NO ff3, NO pycryptodome required anymore.
# All key material and FPE computation now live inside the Azure Function App.
# We only need pandas + requests, both already present in standard Spark images.
import sys, requests, pandas
print("python  :", sys.version.split()[0])
print("requests:", requests.__version__)
print("pandas  :", pandas.__version__)


StatementMeta(, bc2e378f-efc9-499f-ba68-60100879389e, 19, Finished, Available, Finished, False)

ff3 1.0.3
pycryptodome 3.23.0
unicodedata2 17.0.1


In [16]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
#df = spark.read.load.('abfss://bronze@testhmacmaskstor.dfs.core.windows.net/SalesLT.Customer.txt', format='text', header='true')
df = spark.read.option("header",True).csv('abfss://piidata@testlakehouselandingzone.dfs.core.windows.net/SalesLT.Customer.txt')

display(df.limit(10))

StatementMeta(, bc2e378f-efc9-499f-ba68-60100879389e, 20, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 82946515-c569-4093-b9fa-9f3807ac31e3)

In [ ]:
# ---------------------------------------------------------------------------
# Scale the ~1k-row seed CSV up to ~1,000,000 rows for a realistic perf test.
#
# Strategy:
#   * crossJoin with spark.range(MULTIPLIER) -> exactly seed_rows * MULTIPLIER.
#   * Make CustomerID and EmailAddress unique per replica so the encrypted
#     output also has realistic cardinality (FF3 is deterministic, so duplicate
#     inputs would produce duplicate ciphertexts).
#   * Persist as a Delta table in the Lakehouse so subsequent runs can skip
#     this generation step (set REGENERATE = False after the first run).
# ---------------------------------------------------------------------------
from pyspark.sql import functions as F

TARGET_ROWS = 1_000_000
TABLE_NAME  = "Customer_1M"
REGENERATE  = True   # set False after first run to reuse the LH table

if REGENERATE:
    df_seed    = df
    seed_rows  = df_seed.count()
    multiplier = (TARGET_ROWS + seed_rows - 1) // seed_rows  # ceil division
    print(f"seed_rows={seed_rows}  multiplier={multiplier}  -> ~{seed_rows*multiplier:,} rows")

    df_big = (
        df_seed.crossJoin(spark.range(multiplier).withColumnRenamed("id", "_rep"))
               # Keep CustomerID numeric (so the 'numeric' FPE type still applies)
               # by encoding the replica index into a wide-enough integer.
               .withColumn(
                   "CustomerID",
                   (F.col("CustomerID").cast("long") + F.col("_rep") * F.lit(1_000_000)).cast("string"),
               )
               # Make EmailAddress unique per replica (FF3 needs len >= 4 for the
               # local part; the suffix keeps that intact and increases cardinality).
               .withColumn(
                   "EmailAddress",
                   F.concat_ws(
                       "@",
                       F.concat(F.split("EmailAddress", "@").getItem(0), F.lit("_"), F.col("_rep").cast("string")),
                       F.split("EmailAddress", "@").getItem(1),
                   ),
               )
               .drop("_rep")
               .limit(TARGET_ROWS)
    )

    # Pre-repartition before writing so the Delta files are well-sized; this
    # also gives the downstream FPE step a good starting partition count.
    (df_big
        .repartition(64)
        .write.mode("overwrite").format("delta").saveAsTable(TABLE_NAME))

# Use the materialised table as the input for everything below.
df = spark.read.table(TABLE_NAME)
print(f"{TABLE_NAME} row count:", df.count())
display(df.limit(5))


In [ ]:
# Fetch the Azure Function URL + function key from Key Vault.
# The FPE KEY and TWEAK stay inside the Function App (configured there as
# Key Vault references) and are NEVER loaded into the Spark cluster.
from notebookutils import mssparkutils

FPE_FUNCTION_URL = mssparkutils.credentials.getSecret(
    'https://test-lakehouse-kv.vault.azure.net/',
    'fpe-function-url'      # e.g. https://<func-app>.azurewebsites.net/api/fpe
)
FPE_FUNCTION_KEY = mssparkutils.credentials.getSecret(
    'https://test-lakehouse-kv.vault.azure.net/',
    'fpe-function-key'      # the function key (host or function-scoped)
)


StatementMeta(, bc2e378f-efc9-499f-ba68-60100879389e, 21, Finished, Available, Finished, False)

[REDACTED]
[REDACTED]


In [ ]:
# Broadcast small config so every executor gets one copy.
# We never broadcast the FPE key/tweak -- only the Function App URL + function key.
_b_url = spark.sparkContext.broadcast(FPE_FUNCTION_URL)
_b_key = spark.sparkContext.broadcast(FPE_FUNCTION_KEY)

# Tune Arrow batch size: this is exactly the size of each HTTP batch sent to
# the Function App. ~5-10k is a good sweet spot for FF3 in pure Python.
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
spark.conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "5000")


StatementMeta(, bc2e378f-efc9-499f-ba68-60100879389e, 22, Finished, Available, Finished, False)

In [ ]:
"""
Build a scalar pandas_udf for each FPE type.

Per-executor flow:
  1. Spark hands the UDF an Arrow batch as a pandas.Series of strings.
  2. We POST the entire batch (one HTTP call) to the Function App's /api/fpe
     endpoint with {"type": <type>, "values": [...]} and get an array back.
  3. Result is returned as a pandas.Series of the same length and order.

Connection-pooling, retries with exponential back-off, and the requests.Session
are created once per Python worker (module-level singleton inside the closure)
so that subsequent batches reuse TCP/TLS connections.
"""
from typing import Iterator
import json

import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import StringType


def _make_fpe_udf(fpe_type: str):
    url_b = _b_url
    key_b = _b_key

    # Module-scoped singletons keyed per worker -- created lazily on first call.
    def _get_session():
        import threading
        global _FPE_SESSION  # noqa: PLW0603
        try:
            return _FPE_SESSION
        except NameError:
            pass
        import requests
        from requests.adapters import HTTPAdapter
        try:
            from urllib3.util.retry import Retry
        except ImportError:
            from requests.packages.urllib3.util.retry import Retry

        retry = Retry(
            total=5,
            connect=3,
            read=3,
            backoff_factor=0.5,
            status_forcelist=(429, 500, 502, 503, 504),
            allowed_methods=frozenset(["POST"]),
            raise_on_status=False,
            respect_retry_after_header=True,
        )
        adapter = HTTPAdapter(
            pool_connections=32,
            pool_maxsize=32,
            max_retries=retry,
        )
        s = requests.Session()
        s.mount("https://", adapter)
        s.mount("http://", adapter)
        _FPE_SESSION = s  # noqa: F841
        return s

    @pandas_udf(StringType())
    def _udf(it: Iterator[pd.Series]) -> Iterator[pd.Series]:
        sess = _get_session()
        url = url_b.value
        headers = {
            "x-functions-key": key_b.value,
            "Content-Type": "application/json",
        }
        for batch in it:
            # Pandas: NaN -> None for JSON; preserve order.
            values = [None if pd.isna(v) else str(v) for v in batch.tolist()]
            payload = json.dumps({"type": fpe_type, "values": values})
            resp = sess.post(url, data=payload, headers=headers, timeout=(5, 120))
            if resp.status_code != 200:
                raise RuntimeError(
                    f"FPE function call failed: {resp.status_code} {resp.text[:200]}"
                )
            out = resp.json()["values"]
            yield pd.Series(out, index=batch.index, dtype="object")

    return _udf


fpe_ff3_numeric_udf              = _make_fpe_udf("numeric")
fpe_ff3_alphanumeric_udf         = _make_fpe_udf("alphanumeric")
fpe_ff3_alphanumeric_extended_udf = _make_fpe_udf("alphanumeric_extended")
fpe_ff3_phone_udf                = _make_fpe_udf("phone")
fpe_ff3_email_udf                = _make_fpe_udf("email")
fpe_ff3_to_ascii_preserve_other_udf = _make_fpe_udf("ascii_preserve_other")


StatementMeta(, bc2e378f-efc9-499f-ba68-60100879389e, 23, Finished, Available, Finished, False)

In [ ]:
# Smoke test: hit the Function App directly from the driver with a small batch
# of mixed types to confirm connectivity, auth and round-trip semantics
# *before* launching it on a 1M-row DataFrame.
import json, urllib.request

def _http_fpe(fpe_type, values):
    req = urllib.request.Request(
        FPE_FUNCTION_URL,
        data=json.dumps({"type": fpe_type, "values": values}).encode(),
        headers={
            "x-functions-key": FPE_FUNCTION_KEY,
            "Content-Type": "application/json",
        },
        method="POST",
    )
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["values"]

samples = {
    "numeric":              ["12345232"],
    "alphanumeric":         ["Bremer"],
    "alphanumeric_extended": ["Bremer & Sons!, LTD."],
    "phone":                ["06-23112312"],
    "email":                ["bremersons@hotmail.com"],
    "ascii_preserve_other": ["Kožušček123a"],
}
for t, vs in samples.items():
    print(f"{t:25s} {vs[0]!r:30s} => {_http_fpe(t, vs)[0]!r}")


StatementMeta(, bc2e378f-efc9-499f-ba68-60100879389e, 24, Finished, Available, Finished, False)

12345232 => 34237423
Bremer => KIvvfF
Bremer & Sons!, LTD. => w'qdT$k4 WS6MSTostff
06-23112312 => 95-63692799
bremersons@hotmail.com => z6rKTE68és@TDQSllK.com
Kožušček123a => Tlwergik111a


In [ ]:
# Apply the batched, Function-backed UDFs.
#
# Scaling notes for a 1M-row x 7-column DataFrame:
#   * With arrow.maxRecordsPerBatch = 5000 -> 200 HTTP batches per column per
#     task, i.e. 1400 HTTP calls per task across the 7 columns (NOT 7,000,000).
#   * `repartition(N)` controls cluster-side concurrency. Pick N such that the
#     Function App can absorb (N * per-task in-flight) requests. Start with
#     N = (number of executor cores) and scale up only if the Function App
#     reports headroom (CPU < 70%, no 429s).
#   * The Function App should run on Premium / Flex Consumption with
#     pre-warmed instances and high maxBurst; FF3 is CPU-bound pure Python.
N_PARTITIONS = 64    # tune to your cluster + Function App capacity
df_repart = df.repartition(N_PARTITIONS)

df2 = (
    df_repart
      .withColumn('CustomerID',   fpe_ff3_numeric_udf('CustomerID'))
      .withColumn('FirstName',    fpe_ff3_alphanumeric_extended_udf('FirstName'))
      .withColumn('LastName',     fpe_ff3_alphanumeric_extended_udf('LastName'))
      .withColumn('CompanyName',  fpe_ff3_alphanumeric_extended_udf('CompanyName'))
      .withColumn('EmailAddress', fpe_ff3_email_udf('EmailAddress'))
      .withColumn('Phone',        fpe_ff3_phone_udf('Phone'))
      .withColumn('MiddleName',   fpe_ff3_to_ascii_preserve_other_udf('MiddleName'))
)

display(df2.limit(10))


StatementMeta(, bc2e378f-efc9-499f-ba68-60100879389e, 25, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 580436ca-277c-43d8-8223-47e3ca8246b5)

In [22]:
df2.write.mode("overwrite").format("delta").saveAsTable("Customer_FPE")

StatementMeta(, bc2e378f-efc9-499f-ba68-60100879389e, 26, Finished, Available, Finished, True)